python -m pip install langchain-teddynote

In [5]:
from langchain_openai import ChatOpenAI
# ChatOpenAI()는 api_key를 생략하면 현재 Python 환경의 OPENAI_API_KEY를 자동으로 읽습니다
# 이전에 Windows 환경변수에 개인 키를 저장하셨으니, 그 키를 사용했을 가능성이 높습니다
llm = ChatOpenAI(  # 객체 생성
    temperature= 0.1,  # 창의성 (0.0 ~ 2.0)
    model="gpt-5-mini",
)

In [ ]:
# 1.  question = "대한민국의 수도는 어디인가요?"

# 1.  print(f"[답변]: {llm.invoke(question)}")

# completion_tokens 답변에 사용된 토큰수
# prompt_tokens 프롬프트 입력 토큰 수
# total_tokens 전체 토큰수

In [ ]:
question = "대한민국의 수도는 어디인가요 ?"

response = llm.invoke(question)
response

In [ ]:
response.content # 응답 테스트만 출력하고 싶다면 AImessage 객체에서 content 속성에 접근

In [ ]:
# 메타 데이터만 추출하고 싶다면?
response.response_metadata

In [ ]:
response.response_metadata["token_usage"] # token_usage 값만 출력, 실질적으로 토큰이 얼마나 쓰였는지 추적

logprob 활성화

logprob(로그 프로버빌리티)란 주어진 GPT 모델의 토큰 확률 로그 값
즉, 모델이 토큰을 예측할 확률을 나타냄
logprob는 확률 토큰 값에 자연 로그를 씌워서 변환한 값
logprob 값이 0에 가까울수록 확률이 높다, 확률이 낮을수록 음수로 나타남

In [9]:
llm_with_logprob = ChatOpenAI(
    temperature= 0.1,
    max_tokens = 2048,
    model = "gpt-4o-mini",
).bind(llm_logprobs= True)

In [ ]:
question = "대한민국의 수도는 어디인가요 ?"

response = llm_with_logprob.invoke(question)
response.response_metadata

ChatGPT 로 질문을 입력하면 전체 응답이 완성된 뒤에 한 번에 출력되지 않고,
타자로 입력해서 나타내듯이 실시간으로 응답테스트가 출력되는것을 볼 수 있음
이처럼 하나의 토큰 단위로 출력해 주는 기능을 스트리밍 출력이라고 함

In [ ]:
# 스트림 방식으로 질의. answer에 스트리밍 답변의 결과를 받음
answer = llm.stream("대한민국의 아름다운 관광지 10곳과 주소를 알려주세요!")

# 스트리밍 방식으로 각 토큰을 출력(실시간 출력)
for token in answer:
    print(token.content, end="", flush= True)


위 결과는 토큰을 하나씩 생성해서 이어 붙이는 방식으로 출력했을뿐, 
이 상태로는 재활용 할 수 없음, 
이번에는 final_answer 라는 변수를 추가하고 빈 문자열을 할당
그런 다음 반복문 안에서 toekn.content를 final_answer 문자열에 이어 붙여보기

In [14]:
answer = llm.stream("대한민국의 아름다운 관광지 10곳과 주소를 알려주세요")

final_answer = ""
for token in answer:
    final_answer += token.content

In [ ]:
print(final_answer)

반복문으로 스트리밍 출력을 구현하는 대신에 다음과 같이 패키지에서 stream_response를 가져와서 간단하게 구현 가능

In [ ]:
from langchain_teddynote.messages import stream_response


# 스트림 방식으로 질의. answer에 스트리밍 답변의 결과를 받습니다.
answer = llm.stream("대한민국의 아름다운 관광지 10곳과 주소를 알려주세요!")

stream_response(answer)

In [ ]:
answer = llm.stream("대한민국의 아름다운 관광지 10곳과 주소를 알려주세요")
final_answer2 = stream_response(answer, return_output= True)
print(final_answer2)